# 🧹 Actividad 05: Limpieza y Estandarización de Datos
---
**Entrada:** `midagri_limon_raw.csv`, `indeci_eventos_dbf.csv`, `agraria_noticias_raw.csv`  
**Salida:** `midagri_limon_clean.csv`, `indeci_eventos_clean.csv`, `agraria_noticias_clean.csv`

Reglas de limpieza:
- Geografía → **MAYÚSCULAS SIN TILDES** (se conserva Ñ)
- Noticias → Remover HTML, URLs y caracteres especiales con **Regex**
- INDECI → Filtrar solo fenómenos **hidrometeorológicos válidos**


In [ ]:

import os, sys, json, re, warnings, unicodedata
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))
with open('data/02_interim/pipeline_config.json','r',encoding='utf-8') as f:
    CFG = json.load(f)
DIRS=CFG['DIRS']; INTERIM=DIRS['interim']; REPORTS=DIRS['reports']; PROCESSED=DIRS['processed']
print(f"CWD: {os.getcwd()} | Config OK")


## 5.1 Función de Normalización Geográfica

In [ ]:

def norm_geo(t):
    if not isinstance(t, str): return t
    t = t.strip().upper()
    t = t.replace('Ñ','__N__')
    for a,b in [('Á','A'),('É','E'),('Í','I'),('Ó','O'),('Ú','U')]:
        t = t.replace(a,b)
    t = ''.join(c for c in unicodedata.normalize('NFD',t) if unicodedata.category(c)!='Mn')
    return t.replace('__N__','Ñ')

def clean_text(t):
    if not isinstance(t, str): return ''
    t = re.sub(r'<[^>]+>', ' ', t)           # HTML
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)  # URLs
    t = re.sub(r'[^\w\s\.,;:\-\(\)¿\?¡\!áéíóúÁÉÍÓÚñÑ]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

print("Funciones definidas: norm_geo() y clean_text()")


## 5.2 MIDAGRI — Renombrado y Estandarización

In [ ]:

df_m = pd.read_csv(f"{INTERIM}/midagri_limon_raw.csv")
print(f"Shape original: {df_m.shape}")

df_m = df_m.rename(columns={
    'anho':'anho','mes':'mes','COD_UBIGEO':'cod_ubigeo',
    'Dpto':'departamento','Prov':'provincia','Dist':'distrito',
    'dsc_Cultivo':'cultivo','PRODUCCION(t)':'produccion_t',
    'COSECHA (ha)':'cosecha_ha','MTO_PRECCHAC (S/ x kg)':'precio_chacra_kg'
})
for c in ['departamento','provincia','distrito','cultivo']:
    df_m[c] = df_m[c].apply(norm_geo)

df_m['fecha_evento'] = df_m['anho'].astype(str)+'-'+df_m['mes'].astype(str).str.zfill(2)
df_m['produccion_t']     = pd.to_numeric(df_m['produccion_t'], errors='coerce').fillna(0)
df_m['cosecha_ha']       = pd.to_numeric(df_m['cosecha_ha'], errors='coerce').fillna(0)
df_m['precio_chacra_kg'] = pd.to_numeric(df_m['precio_chacra_kg'], errors='coerce')

out = f"{INTERIM}/midagri_limon_clean.csv"
df_m.to_csv(out, index=False, encoding='utf-8-sig')
print(f"Dptos únicos: {df_m['departamento'].nunique()} | Provincias: {df_m['provincia'].nunique()}")
print(f"[OK] {out}")
print(df_m[['fecha_evento','departamento','provincia','produccion_t','precio_chacra_kg']].head(3).to_string(index=False))


## 5.3 INDECI — Filtro de Fenómenos + Estandarización

In [ ]:

df_ev = pd.read_csv(f"{INTERIM}/indeci_eventos_dbf.csv", low_memory=False)
print(f"Eventos originales: {len(df_ev):,}")

# Renombrar columnas clave
rename_map = {'departamen':'departamento','provincia':'provincia','fenomeno':'fenomeno'}
df_ev = df_ev.rename(columns={k:v for k,v in rename_map.items() if k in df_ev.columns})
for c in ['departamento','provincia','fenomeno']:
    if c in df_ev.columns:
        df_ev[c] = df_ev[c].apply(norm_geo)

df_ev['fecha'] = pd.to_datetime(df_ev['fecha'], errors='coerce')
df_ev['fecha_evento'] = df_ev['fecha'].dt.strftime('%Y-%m')

num_map = {'safecta':'personas_afectadas','sdamni':'personas_damnificadas',
           'sareaculti':'has_cultivo_afectadas','sareacul_1':'has_cultivo_perdidas'}
for s,d in num_map.items():
    if s in df_ev.columns:
        df_ev[d] = pd.to_numeric(df_ev[s], errors='coerce').fillna(0)

PELIGROS = [p.upper() for p in CFG['PELIGROS_VALIDOS']]
df_clean = df_ev[df_ev['fenomeno'].isin(PELIGROS)].copy()
print(f"Eventos hidrometeorológicos: {len(df_clean):,} ({len(df_clean)/len(df_ev)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(10,4))
kept = df_clean['fenomeno'].value_counts()
kept.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Fenómenos Válidos tras Filtro', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig(f"{REPORTS}/g6_indeci_filtro.png", dpi=150); plt.show()

out = f"{INTERIM}/indeci_eventos_clean.csv"
df_clean.to_csv(out, index=False, encoding='utf-8-sig')
print(f"[OK] {out}")


## 5.4 AGRARIA.PE — Limpieza NLP con Regex

In [ ]:

df_n = pd.read_csv(f"{INTERIM}/agraria_noticias_raw.csv")
df_n['fecha'] = pd.to_datetime(df_n['fecha'], errors='coerce')
df_n['fecha_evento'] = df_n['fecha'].dt.strftime('%Y-%m')

df_n['titular_clean'] = df_n['titular'].apply(clean_text)
df_n['cuerpo_clean']  = df_n['cuerpo_completo'].apply(clean_text)

orig_len  = df_n['cuerpo_completo'].astype(str).str.len().mean()
clean_len = df_n['cuerpo_clean'].str.len().mean()
print(f"Longitud media original:  {orig_len:,.0f} chars")
print(f"Longitud media limpia:    {clean_len:,.0f} chars  (reducción {(1-clean_len/orig_len)*100:.1f}%)")

# Comparación antes/después — primera noticia
idx = df_n['cuerpo_completo'].astype(str).str.len().idxmax()
print(f"\nEjemplo — Antes (primeros 200 chars):")
print(str(df_n.loc[idx,'cuerpo_completo'])[:200])
print(f"\nEjemplo — Después (primeros 200 chars):")
print(df_n.loc[idx,'cuerpo_clean'][:200])

out = f"{INTERIM}/agraria_noticias_clean.csv"
df_n.to_csv(out, index=False, encoding='utf-8-sig')
print(f"\n[OK] {out} ({len(df_n)} filas)")
print("[ACTIVIDAD 05] COMPLETADA.")


## TODO: INTEGRACIÓN DATA NASA (COMPAÑERO)
Normalizar al integrar:
```python
df_nasa = pd.read_csv(f"{INTERIM}/nasa_clima_raw.csv")
df_nasa['departamento'] = df_nasa['departamento'].apply(norm_geo)  # misma función
df_nasa['fecha_evento'] = pd.to_datetime(df_nasa['DATE']).dt.strftime('%Y-%m')
# Validar coordenadas dentro de Perú
valid = df_nasa['lat'].between(-18.5,-0.1) & df_nasa['lon'].between(-81.4,-68.7)
print(f"Registros fuera de Perú: {(~valid).sum()}")
```
